# 대화 이력 관리를 위한 메모리 구현 (Chat History)

### **학습 목표**
1. LangChain의 다양한 메모리 관리 방식 이해 (레거시 vs 최신)
2. RunnableWithMessageHistory를 활용한 대화 이력 관리 (레거시 방식)
3. LangChain 1.0의 `create_agent` + Checkpointer를 활용한 메모리 구현 (권장)
4. 메시지 트리밍과 대화 요약을 통한 컨텍스트 윈도우 관리

---

## 환경 설정 및 준비

### 사전 준비

**필수 환경 변수:**
`.env` 파일에 다음 내용을 추가하세요:
```
OPENAI_API_KEY=sk-...
```

**필수 패키지:**
```bash
uv add langchain langchain-openai langgraph langgraph-checkpoint-sqlite
```

**참고:**
- LangChain 1.0부터 `create_agent`가 에이전트 생성의 표준 방식입니다
- `create_agent`는 내부적으로 LangGraph 기반으로 동작합니다

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from pprint import pprint

`(3) LLM 설정`

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-nano',      # 사용할 모델
    temperature=0.3,            # 응답의 창의성 조절 (0~1, 낮을수록 일관적)
    top_p=0.9,                  # 토큰 샘플링 확률 임계값 (0~1)
)

e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from langfuse.langchain import CallbackHandler 

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

#config={"callbacks": [langfuse_handler]}

## 기본 개념: 단기 메모리 vs 장기 메모리

- **단기 메모리 (Short-term Memory)**
    - **Thread 기반 대화 관리**: 각 대화 세션(thread)별로 메시지 히스토리를 유지
    - **Checkpointer**를 통해 에이전트 실행 시점마다 상태를 저장
    - 같은 `thread_id`를 사용하면 이전 대화 내용을 기억

- **장기 메모리 (Long-term Memory)**
    - **Store**를 통해 thread를 넘어서 정보를 공유
    - 사용자별 선호도, 프로필 정보 등을 저장
    - 여러 대화 세션에서 공통으로 접근 가능

- **LangChain 1.0 권장 패턴**
    - `create_agent`가 에이전트 생성의 표준 방식
    - 내부적으로 LangGraph 기반으로 동작
    - `checkpointer` 파라미터로 메모리 관리 자동화

---

## 1. 메시지 전달 방식 (Message Passing)

* 메시지 전달 방식은 LangChain에서 가장 기본적인 메모리 구현 방법으로, 이전 대화 기록(chat history)을 체인에 직접 전달하여 문맥을 유지하는 방식입니다.

* 이 방식은 SystemMessage(시스템 지시사항), HumanMessage(사용자 입력), AIMessage(AI 응답) 등 다양한 유형의 메시지를 ChatPromptTemplate을 통해 구조화하며, MessagesPlaceholder를 사용하여 이전 대화 내용을 포함시킵니다.

* 챗봇의 기본적인 메모리 시스템을 구현하는데 사용되며, 이를 통해 AI는 이전 대화 맥락을 이해하고 그에 맞는 적절한 응답을 생성할 수 있습니다.

In [4]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ChatPromptTemplate를 사용하여 챗봇의 초기 메시지를 정의
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant."),
    MessagesPlaceholder(variable_name="messages"),    # 메시지 목록을 동적으로 삽입하는 부분
])

# ChatPromptTemplate에 삽입할 메시지 목록을 정의
messages = [
        HumanMessage(content="안녕하세요. 제 이름은 홍길동입니다."),
        AIMessage(content="안녕하세요! 어떻게 도와드릴까요?"),
]    

# ChatPromptTemplate에 삽입할 메시지 목록을 업데이트하고 출력 
pprint(prompt.format(messages=messages))

('System: You are a helpful assistant.\n'
 'Human: 안녕하세요. 제 이름은 홍길동입니다.\n'
 'AI: 안녕하세요! 어떻게 도와드릴까요?')


In [5]:
# 대화형 체인을 정의 (prompt -> llm)
chain = prompt | llm

# 기본적인 메시지 전달: 이전 메시지 목록에 새로운 메시지를 추가해서 전달
response = chain.invoke({
    "messages": messages + [HumanMessage(content="제 이름을 기억하나요?")] # 이전 메시지를 기억하는지 확인하는 질문 메시지 추가
})

pprint(response)

AIMessage(content='네, 홍길동님이라고 하셨죠! 기억하고 있겠습니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 53, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_065579f22e', 'id': 'chatcmpl-DsShsCzMhWPaHzapFvKALqFEFos1d', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019edfdf-6a40-76e3-958e-b94c4a1fc1a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 24, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})


---

## 2. RunnableWithMessageHistory (레거시 방식)

* `RunnableWithMessageHistory`는 LangChain에서 대화 기록을 관리하는 고급 기능으로, 체인의 실행 과정에서 메시지 기록을 자동으로 저장하고 검색할 수 있게 해주는 래퍼(wrapper) 클래스입니다.

* 이 기능은 대화 세션별로 독립적인 기록을 유지할 수 있게 해주며, `ConfigurableField`를 통해 메모리 구성을 유연하게 조정할 수 있습니다.

* 구현 시에는 `get_session_history` 콜백을 통해 세션ID별로 메시지 기록을 관리하며, 이를 통해 각 대화의 컨텍스트를 정확하게 유지할 수 있습니다.

    | 저장소 유형 | 클래스 | 용도 |
    |------------|--------|------|
    | 메모리 기반 | `InMemoryHistory` (커스텀) | 개발/테스트용, 휘발성 |
    | SQLite | `SQLiteChatMessageHistory` (커스텀) | 로컬 영구 저장 |

### 2.1 InMemoryHistory (메모리 기반) 구현

* `InMemoryHistory` 클래스는 대화 이력의 기본 구조를 제공하며, `BaseChatMessageHistory`와 `BaseModel`을 상속받아 메시지를 메모리에서 효율적으로 관리합니다.

* `store` 변수는 전역 딕셔너리로 구현되어 세션별 대화 이력을 구분하여 저장합니다. `session_id`를 키로 사용하여 각 세션의 `InMemoryHistory` 객체에 빠르게 접근할 수 있습니다.

* `get_session_history` 함수는 세션 관리의 진입점 역할을 하며, 존재하지 않는 세션에 대해 자동으로 새로운 `InMemoryHistory` 객체를 생성하는 팩토리 패턴을 구현합니다.

In [6]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field
from typing import List

# 메모리 기반 히스토리 구현
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    """메모리 기반 대화 히스토리 저장소"""
    messages: List[BaseMessage] = Field(default_factory=list)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """메시지 추가"""
        self.messages.extend(messages)
    
    def clear(self) -> None:
        """히스토리 초기화"""
        self.messages = []

# 세션 저장소 (전역 딕셔너리)
store = {}

# 세션 ID로 히스토리 가져오기
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """세션 ID에 해당하는 히스토리 반환 (없으면 새로 생성)"""
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

print("InMemoryHistory 클래스가 정의되었습니다.")

InMemoryHistory 클래스가 정의되었습니다.


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 프롬프트 템플릿 설정
prompt_legacy = ChatPromptTemplate.from_messages([
    ("system", "당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요."),
    MessagesPlaceholder(variable_name="history"), 
    ("human", "{input}")
])

# 프롬프트와 llm을 연결하여 체인 생성
chain_legacy = prompt_legacy | llm

# 히스토리 관리 추가  
chain_with_history = RunnableWithMessageHistory(
    chain_legacy,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 지정된 세션 ID(tourist_legacy_1)를 사용하여 체인 실행
response = chain_with_history.invoke(
    {"input": "서울에서 가볼만한 곳을 추천해주세요."},
    config={"configurable": {"session_id": "tourist_legacy_1"}, "callbacks": [langfuse_handler]}
)

print("여행 가이드 답변:")
print(response.content)

여행 가이드 답변:
서울은 역사와 현대 문화가 어우러진 도시로, 다양한 명소와 체험거리를 즐기실 수 있습니다. 아래 추천드리는 곳들을 참고해보세요:

1. 경복궁과 광화문 광장  
한국의 대표 궁궐인 경복궁은 조선시대의 아름다움을 느낄 수 있는 곳입니다. 광화문 광장과 함께 방문하면 한국의 역사를 체험할 수 있습니다. 한복 체험도 추천드립니다.

2. 북촌 한옥마을  
전통 한옥이 잘 보존된 마을로, 고즈넉한 분위기 속에서 한국 전통 가옥과 골목길을 산책하며 옛 정취를 느껴보세요.

3. 남산서울타워  
서울의 랜드마크인 남산타워에서 도심 전경을 감상하세요. 케이블카를 타거나 산책로를 따라 오를 수 있으며, 야경이 특히 아름답습니다.

4. 명동과 동대문 디자인 플라자(DDP)  
쇼핑과 맛집이 가득한 명동 거리와 현대적인 건축물인 DDP는 꼭 방문해볼 만한 곳입니다. 밤에는 조명이 아름다워 사진 찍기 좋아요.

5. 홍대 거리  
젊음과 예술이 넘치는 거리로, 라이브 공연, 독특한 카페, 스트리트 아트 등을 즐기실 수 있습니다. 활기찬 분위기를 만끽하세요.

6. 한강공원  
한강을 따라 조성된 공원에서 자전거 타기, 피크닉, 산책 등 야외 활동을 즐기실 수 있습니다. 여름철에는 야시장과 수상 스포츠도 인기입니다.

7. 인사동과 삼청동  
전통과 현대가 어우러진 거리로, 전통 공예품과 예술 작품, 맛집이 가득합니다. 한국 전통문화 체험도 가능합니다.

이 외에도 서울에는 박물관, 미술관, 맛집 등 볼거리와 즐길거리가 많으니 일정과 취향에 맞게 여행을 계획해보세요! 궁금한 점이 있거나 더 구체적인 추천이 필요하시면 언제든 말씀해 주세요.


In [13]:
# 대화 히스토리 출력
history = get_session_history("tourist_legacy_1")
pprint(history.messages)

[HumanMessage(content='서울에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='서울은 역사와 현대가 어우러진 매력적인 도시로, 다양한 명소를 즐기실 수 있습니다. 아래 추천드리는 곳들을 참고하세요:\n\n1. 경복궁\n조선시대의 대표 궁궐로, 아름다운 건축물과 함께 전통 의상 체험도 가능합니다. 근처에 위치한 국립민속박물관도 함께 방문해보세요.\n\n2. 북촌 한옥마을\n전통 한옥이 잘 보존된 마을로, 고즈넉한 분위기 속에서 한국의 전통 가옥과 골목길을 산책할 수 있습니다.\n\n3. 명동\n서울의 대표 쇼핑 거리로, 패션, 화장품, 길거리 음식 등을 즐기실 수 있습니다. 야경이 아름다운 명동성당도 추천드립니다.\n\n4. 남산서울타워\n도심 속 자연과 함께 서울의 전경을 한눈에 내려다볼 수 있는 명소입니다. 케이블카를 타고 올라가거나 산책로를 따라 오를 수 있습니다.\n\n5. 홍대 거리\n젊음의 거리로, 예술과 음악, 독특한 카페와 상점들이 가득합니다. 밤에는 라이브 공연도 즐기실 수 있어 활기찬 분위기를 만끽하세요.\n\n6. 동대문 디자인 플라자(DDP)\n현대적인 건축물과 다양한 전시, 패션 시장이 결합된 복합문화공간입니다. 야간 조명도 아름다워 사진 찍기 좋은 곳입니다.\n\n7. 한강공원\n한강을 따라 조성된 공원으로, 자전거 타기, 피크닉, 산책 등 다양한 야외 활동을 즐기실 수 있습니다.\n\n이 외에도 서울에는 맛집, 박물관, 미술관 등 다양한 즐길 거리와 볼거리가 많으니, 일정과 취향에 맞게 여행 계획을 세워보세요! 궁금한 점이 있거나 더 구체적인 추천이 필요하시면 언제든 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 442, 'prompt_tokens': 40, 'total_token

In [15]:
# 이전 대화 내용을 기반으로 후속 질문
response = chain_with_history.invoke(
    {"input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"},
    config={"configurable": {"session_id": "tourist_legacy_1"}, "callbacks": [langfuse_handler]}
)

print("여행 가이드 답변:")
print(response.content)

여행 가이드 답변:
이전에 추천드린 장소 중에서 가장 인기 있는 곳은 **경복궁**과 **남산서울타워**입니다.

- **경복궁**은 한국의 대표 궁궐로, 한국 전통 건축과 역사를 체험할 수 있는 최고의 명소입니다. 특히, 한복 체험과 근처 북촌 한옥마을, 인사동과 연계해서 방문하면 더욱 풍성한 경험을 하실 수 있습니다.

- **남산서울타워**는 서울의 상징적인 랜드마크로, 도심 전경과 야경이 매우 아름다워 많은 관광객들이 찾는 곳입니다. 케이블카를 타고 올라가거나 산책로를 따라 오를 수 있어 인기가 높습니다.

이 두 곳은 서울 여행의 필수 코스로 손꼽히며, 방문객들이 가장 많이 찾는 명소입니다. 여행 일정에 꼭 포함시켜보시길 추천드려요!


### 2.2 SQLiteChatMessageHistory (영구 저장소)

* SQLite 통합 구현을 위해서는 먼저 메시지를 저장할 데이터베이스 테이블 구조를 정의하고, `BaseChatMessageHistory`를 상속받아 메시지 저장/조회 로직을 구현해야 합니다.

* 세션 ID를 기준으로 대화 내용을 구분하여 관리하며, 메시지의 타입(Human/AI), 내용, 메타데이터, 타임스탬프 등의 정보를 체계적으로 저장합니다.

* 프로세스가 재시작되어도 대화 내용이 유지되어, 프로덕션 환경에서 사용하기 적합합니다.

In [16]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
import sqlite3
from typing import List
import json

class SQLiteChatMessageHistory(BaseChatMessageHistory):
    """
    SQLite 데이터베이스를 사용하여 챗봇 대화 히스토리를 저장하는 클래스

    Attributes:
        session_id (str): 세션 ID
        db_path (str): SQLite 데이터베이스 파일 경로
    """
    def __init__(self, session_id: str, db_path: str = "chat_history_legacy.db"):
        self.session_id = session_id
        self.db_path = db_path
        self._create_tables()
    
    def _create_tables(self):
        """데이터베이스 테이블 생성"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT,
                message_type TEXT,
                content TEXT,
                metadata TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        
        conn.commit()
        conn.close()
    
    def add_message(self, message: BaseMessage) -> None:
        """단일 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            INSERT INTO messages (session_id, message_type, content, metadata)
            VALUES (?, ?, ?, ?)
        """, (
            self.session_id,
            message.__class__.__name__,
            message.content,
            json.dumps(message.additional_kwargs)
        ))
        
        conn.commit()
        conn.close()
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """여러 메시지 추가"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        for message in messages:
            cursor.execute("""
                INSERT INTO messages (session_id, message_type, content, metadata)
                VALUES (?, ?, ?, ?)
            """, (
                self.session_id,
                message.__class__.__name__,
                message.content,
                json.dumps(message.additional_kwargs)
            ))
        
        conn.commit()
        conn.close()
    
    def clear(self) -> None:
        """세션의 모든 메시지 삭제"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            DELETE FROM messages WHERE session_id = ?
        """, (self.session_id,))
        
        conn.commit()
        conn.close()
    
    @property
    def messages(self) -> List[BaseMessage]:
        """저장된 메시지 조회"""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT message_type, content, metadata
            FROM messages 
            WHERE session_id = ?
            ORDER BY created_at
        """, (self.session_id,))
        
        messages = []
        for row in cursor.fetchall():
            message_type, content, metadata = row
            if message_type == "HumanMessage":
                message = HumanMessage(content=content)
            else:
                message = AIMessage(content=content)
            
            if metadata:
                message.additional_kwargs = json.loads(metadata)
            
            messages.append(message)
        
        conn.close()
        return messages

# 세션 ID로 히스토리 가져오기
def get_sqlite_history(session_id: str) -> BaseChatMessageHistory:
    return SQLiteChatMessageHistory(session_id=session_id)

print("SQLiteChatMessageHistory 클래스가 정의되었습니다.")

SQLiteChatMessageHistory 클래스가 정의되었습니다.


In [18]:
# SQLite 히스토리를 사용하는 RunnableWithMessageHistory 체인 구성
chain_with_sqlite = RunnableWithMessageHistory(
    chain_legacy,
    get_sqlite_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 테스트 대화
response = chain_with_sqlite.invoke(
    {"input": "수원에서 가볼만한 곳을 추천해주세요."},
    config={"configurable": {"session_id": "tourist_sqlite_1"}}
)

print("여행 가이드 답변:")
print(response.content)

여행 가이드 답변:
수원은 역사와 문화가 풍부한 도시로, 다양한 명소들이 있습니다. 여행객에게 추천드릴 만한 곳은 다음과 같습니다:

1. 수원 화성 (수원 화성 성곽)
   - 조선 시대의 대표적인 성곽으로 유네스코 세계문화유산에 등재되어 있습니다.
   - 성곽을 따라 산책하며 아름다운 경관과 역사적인 건축물을 감상할 수 있습니다.
   - 화성행궁과 함께 방문하면 더욱 풍성한 체험이 가능합니다.

2. 화성행궁
   - 조선 시대 왕실의 별궁으로, 화성 내에 위치해 있습니다.
   - 궁궐 내부를 둘러보며 조선 왕실의 생활상을 엿볼 수 있습니다.
   - 야간에는 조명이 켜져 더욱 아름다운 야경을 즐기실 수 있습니다.

3. 수원 전통시장 (수원 중앙시장)
   - 지역 특산품과 다양한 먹거리를 즐길 수 있는 곳입니다.
   - 수원만의 맛집과 전통 공예품 등을 구경하며 현지 문화를 체험하세요.

4. 광교호수공원
   - 현대적인 도시공원으로, 산책이나 자전거 타기에 적합합니다.
   - 호수 주변의 자연경관과 함께 휴식을 취하기 좋은 장소입니다.

5. 수원 박물관
   - 수원과 관련된 역사와 문화를 배울 수 있는 곳입니다.
   - 다양한 전시와 체험 프로그램이 마련되어 있습니다.

6. 팔달문과 수원역 주변
   - 전통과 현대가 어우러진 거리로, 카페와 레스토랑, 쇼핑 장소가 많아 산책하기 좋습니다.

수원은 역사와 자연, 현대적인 도시 문화가 조화를 이루는 곳이니, 일정에 맞춰 여러 명소를 방문해보시길 추천드립니다. 즐거운 여행 되세요!


In [19]:
# SQLite에 저장된 대화 히스토리 확인
history = get_sqlite_history("tourist_sqlite_1")
pprint(history.messages)

[HumanMessage(content='수원에서 가볼만한 곳을 추천해주세요.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='수원은 역사와 문화가 풍부한 도시로, 다양한 명소들이 있습니다. 여행객에게 추천드릴 만한 곳은 다음과 같습니다:\n\n1. 수원 화성 (수원 화성 성곽)\n   - 조선 시대의 대표적인 성곽으로 유네스코 세계문화유산에 등재되어 있습니다.\n   - 성곽을 따라 산책하며 아름다운 경관과 역사적인 건축물을 감상할 수 있습니다.\n   - 화성행궁과 함께 방문하면 더욱 풍성한 체험이 가능합니다.\n\n2. 화성행궁\n   - 조선 시대 왕실의 별궁으로, 화성 내에 위치해 있습니다.\n   - 궁궐 내부를 둘러보며 조선 왕실의 생활상을 엿볼 수 있습니다.\n   - 야간에는 조명이 켜져 더욱 아름다운 야경을 즐기실 수 있습니다.\n\n3. 수원 전통시장 (수원 중앙시장)\n   - 지역 특산품과 다양한 먹거리를 즐길 수 있는 곳입니다.\n   - 수원만의 맛집과 전통 공예품 등을 구경하며 현지 문화를 체험하세요.\n\n4. 광교호수공원\n   - 현대적인 도시공원으로, 산책이나 자전거 타기에 적합합니다.\n   - 호수 주변의 자연경관과 함께 휴식을 취하기 좋은 장소입니다.\n\n5. 수원 박물관\n   - 수원과 관련된 역사와 문화를 배울 수 있는 곳입니다.\n   - 다양한 전시와 체험 프로그램이 마련되어 있습니다.\n\n6. 팔달문과 수원역 주변\n   - 전통과 현대가 어우러진 거리로, 카페와 레스토랑, 쇼핑 장소가 많아 산책하기 좋습니다.\n\n수원은 역사와 자연, 현대적인 도시 문화가 조화를 이루는 곳이니, 일정에 맞춰 여러 명소를 방문해보시길 추천드립니다. 즐거운 여행 되세요!', additional_kwargs={'refusal': None}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [20]:
# 후속 질문 - SQLite에서 이전 대화를 불러옴
response = chain_with_sqlite.invoke(
    {"input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"},
    config={"configurable": {"session_id": "tourist_sqlite_1"}}
)

print("여행 가이드 답변:")
print(response.content)

여행 가이드 답변:
수원에서 가장 인기 있는 장소는 바로 **수원 화성**입니다. 

수원 화성은 조선 시대의 대표적인 성곽으로, 유네스코 세계문화유산에 등재되어 있어 국내외 관광객 모두에게 매우 유명합니다. 성곽을 따라 걷거나 성곽 내외의 다양한 포토존에서 사진을 찍으며 역사적인 분위기를 만끽할 수 있습니다. 특히, 화성행궁과 함께 방문하면 조선 왕실의 아름다움과 역사를 깊이 체험할 수 있어 많은 사람들이 찾는 명소입니다.

이곳은 역사적 가치뿐만 아니라 아름다운 자연경관과 잘 보존된 건축물 덕분에 수원 여행의 하이라이트로 손꼽히며, 방문객들에게 잊지 못할 추억을 선사합니다. 

수원에 오시면 꼭 한 번 방문해보시길 추천드립니다!


> **참고**: `RunnableWithMessageHistory`는 LangChain 1.0에서도 여전히 동작하지만, 새로운 프로젝트에서는 `create_agent` + `checkpointer` 방식을 권장합니다. 이 섹션에서는 레거시 방식의 내부 동작 원리를 이해하기 위해 학습합니다.

### 2.3 메시지 트리밍 (TrimmedInMemoryHistory)

* 대화가 길어지면 컨텍스트 윈도우 제한에 도달할 수 있습니다. `TrimmedInMemoryHistory` 클래스는 메시지를 추가할 때 자동으로 오래된 메시지를 제거하여 최근 메시지만 유지합니다.

* `trim_messages` 함수를 활용하여 지정된 개수의 메시지만 유지하며, "last" 전략을 사용하여 가장 최근의 메시지부터 보존합니다.

* 이 방식은 메모리 효율성과 컨텍스트 품질의 균형을 맞추는 데 유용합니다.

In [ ]:
from langchain_core.messages import trim_messages

# 메시지 트리밍이 적용된 인메모리 히스토리 구현
class TrimmedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    """메시지 트리밍이 적용된 메모리 기반 히스토리"""
    messages: List[BaseMessage] = Field(default_factory=list)
    max_tokens: int = Field(default=4)  # 유지할 최대 메시지 수
    
    def __init__(self, max_tokens: int = 4, **kwargs):
        super().__init__(max_tokens=max_tokens, **kwargs)
    
    def add_messages(self, messages: List[BaseMessage]) -> None:
        """메시지 추가 후 트리밍 수행"""
        self.messages.extend(messages)
        # 메시지 추가 후 트리밍 수행
        trimmer = trim_messages(
            strategy="last",
            max_tokens=self.max_tokens,
            token_counter=len  # 메시지 개수 기준
        )
        self.messages = trimmer.invoke(self.messages)
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
trimmed_store = {}

# 세션 ID로 트리밍된 히스토리 가져오기
def get_trimmed_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in trimmed_store:
        trimmed_store[session_id] = TrimmedInMemoryHistory(max_tokens=4)
    return trimmed_store[session_id]

print("TrimmedInMemoryHistory 클래스가 정의되었습니다.")

TrimmedInMemoryHistory 클래스가 정의되었습니다.


In [22]:
# 트리밍된 히스토리를 사용하는 체인 구성
chain_with_trimmed = RunnableWithMessageHistory(
    chain_legacy,
    get_trimmed_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 첫 번째 질문
response = chain_with_trimmed.invoke(
    {"input": "서울에서 가볼만한 곳을 추천해주세요."},
    config={"configurable": {"session_id": "tourist_trimmed_1"}}
)

print("여행 가이드 답변:")
print(response.content)

여행 가이드 답변:
서울은 역사와 현대가 어우러진 매력적인 도시로, 다양한 명소를 즐기실 수 있습니다. 아래 추천드리는 곳들을 참고하세요!

1. 경복궁
조선시대의 대표 궁궐로, 아름다운 건축물과 함께 전통 의상 체험(한복 대여)도 즐기실 수 있습니다. 근처 북촌 한옥마을도 함께 방문하면 전통 한옥의 멋을 느낄 수 있습니다.

2. 명동
서울의 대표적인 쇼핑 거리로, 패션, 화장품, 길거리 음식 등을 즐기기에 좋습니다. 밤에는 화려한 네온사인과 활기찬 분위기를 만끽하세요.

3. 남산서울타워
남산 정상에 위치한 전망대로, 서울 시내 전경을 한눈에 내려다볼 수 있습니다. 케이블카를 타고 올라가거나 산책로를 따라 걷는 것도 추천합니다.

4. 인사동
전통과 현대가 어우러진 거리로, 전통 공예품, 갤러리, 찻집, 전통 음식점이 많아 문화 체험에 적합합니다.

5. 홍대
젊음과 예술의 거리로, 거리 공연, 독특한 카페, 개성 넘치는 상점들이 많아 활기찬 분위기를 즐기실 수 있습니다.

6. 한강공원
한강을 따라 조성된 공원으로, 자전거 타기, 피크닉, 유람선 타기 등 다양한 야외 활동이 가능합니다.

7. 동대문 디자인 플라자(DDP)
현대적인 건축물로, 패션 전시회와 다양한 이벤트가 열리며, 야경도 매우 아름답습니다.

이 외에도 서울에는 박물관, 미술관, 시장 등 다양한 명소가 있으니 일정과 관심사에 맞게 방문 계획을 세우시면 좋겠습니다. 즐거운 여행 되세요!


In [23]:
# 두 번째 질문
response = chain_with_trimmed.invoke(
    {"input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"},
    config={"configurable": {"session_id": "tourist_trimmed_1"}}
)

print("여행 가이드 답변:")
print(response.content)
print(f"\n현재 메시지 수: {len(get_trimmed_history('tourist_trimmed_1').messages)}")

여행 가이드 답변:
가장 인기 있는 장소는 바로 **경복궁**입니다. 

경복궁은 조선시대의 대표 궁궐로서, 한국의 역사와 전통을 느낄 수 있는 최고의 명소입니다. 특히, 광화문 광장과 근정전, 경회루 등 아름다운 건축물과 정원이 인상적이며, 전통 의상인 한복을 입고 방문하면 더욱 특별한 경험을 하실 수 있습니다. 많은 관광객들이 역사 체험과 사진 촬영을 위해 찾는 곳으로, 서울 여행의 필수 코스입니다.

이외에도, 경복궁 근처 북촌 한옥마을과 인사동도 함께 방문하면 한국 전통 문화의 정취를 만끽하실 수 있습니다.

현재 메시지 수: 4


In [24]:
# 세 번째 질문 - 트리밍 효과 확인 (max_tokens=4이므로 오래된 메시지 제거됨)
response = chain_with_trimmed.invoke(
    {"input": "그 장소의 입장료는 얼마인가요?"},
    config={"configurable": {"session_id": "tourist_trimmed_1"}}
)

print("여행 가이드 답변:")
print(response.content)

# 트리밍된 히스토리 확인
print("\n[트리밍된 대화 히스토리]")
history = get_trimmed_history("tourist_trimmed_1")
for i, msg in enumerate(history.messages):
    role = msg.__class__.__name__.replace("Message", "")
    content = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"{i+1}. [{role}]: {content}")

여행 가이드 답변:
경복궁의 입장료는 일반적으로 성인 기준으로 **3,000원**입니다. 

단, 만 24세 이하의 학생이나 국가유공자, 장애인 등은 할인 혜택이 적용되어 더 저렴하게 입장하실 수 있습니다. 또한, 만 7세 이하 어린이와 65세 이상 노인도 무료 입장이 가능합니다.

참고로, 특별 전시나 야간 개장 시에는 별도 요금이 부과될 수 있으니 방문 전에 공식 홈페이지 또는 현장 안내를 확인하시는 것이 좋습니다. 

즐거운 방문 되시길 바랍니다!

[트리밍된 대화 히스토리]
1. [Human]: 이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?
2. [AI]: 가장 인기 있는 장소는 바로 **경복궁**입니다. 

경복궁은 조선시대의 대표 궁궐로서, 한국의 역사와 전통...
3. [Human]: 그 장소의 입장료는 얼마인가요?
4. [AI]: 경복궁의 입장료는 일반적으로 성인 기준으로 **3,000원**입니다. 

단, 만 24세 이하의 학생이나 국...


### 2.4 대화 요약 저장 (SummarizedInMemoryHistory)

* 대화가 길어질 경우, 전체 대화 내용을 요약하여 컨텍스트로 활용하는 방식으로 이전 대화의 핵심을 추출합니다.

* 메시지 히스토리가 지정된 길이(예: 6개의 메시지)를 초과할 경우, 이전 대화들을 요약하고 가장 최근의 메시지만 유지하는 방식으로 새로운 대화 히스토리를 구성합니다.

* 이러한 요약 메모리 방식을 통해 토큰 사용량을 크게 줄이면서도 대화의 핵심 문맥을 유지할 수 있으며, 특히 장시간 진행되는 대화에서 효과적입니다.

In [25]:
from langchain_core.messages import SystemMessage

class SummarizedInMemoryHistory(BaseChatMessageHistory, BaseModel):
    """대화 요약이 적용된 메모리 기반 히스토리"""
    messages: List[BaseMessage] = Field(default_factory=list)
    summary_threshold: int = Field(default=6)  # 요약을 시작할 메시지 수
    llm: ChatOpenAI = Field(default_factory=lambda: ChatOpenAI(
        model="gpt-4.1-mini", temperature=0.1, top_p=0.9
    ))
    
    def add_messages(self, new_messages: List[BaseMessage]) -> None:
        """메시지 추가 및 필요시 요약 수행"""
        self.messages.extend(new_messages)
        
        print(f"현재 메시지 수: {len(self.messages)}")
        
        # 메시지 수가 임계값을 넘으면 요약 수행
        if len(self.messages) >= self.summary_threshold:
            # 마지막 사용자/AI 메시지 쌍 저장
            last_user_message = self.messages[-2]
            last_ai_message = self.messages[-1]
            
            # 요약 생성
            summary_prompt = (
                "Distill the above chat messages into a single summary message. "
                "Include as many specific details as you can. "
                "Use the original language and tone of the conversation."
            )
            
            summary_chain_messages = [
                SystemMessage(content=(
                    "You are a helpful assistant. "
                    "Your task is to summarize the conversation accurately."
                )),
                *self.messages[:-2],  # 마지막 대화 턴을 제외한 모든 메시지
                HumanMessage(content=summary_prompt)
            ]
            
            # 요약 생성
            summary = self.llm.invoke(summary_chain_messages)
            
            # 메시지 리스트 초기화 후 요약과 마지막 메시지 추가
            self.messages = [
                summary,
                last_user_message,
                last_ai_message
            ]
            print("→ 대화가 요약되었습니다.")
    
    def clear(self) -> None:
        self.messages = []

# 세션 저장소
summarized_store = {}

# 세션 ID로 요약된 히스토리 가져오기
def get_summarized_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in summarized_store:
        summarized_store[session_id] = SummarizedInMemoryHistory(summary_threshold=6)
    return summarized_store[session_id]

print("SummarizedInMemoryHistory 클래스가 정의되었습니다.")

SummarizedInMemoryHistory 클래스가 정의되었습니다.


In [26]:
# 요약된 히스토리를 사용하는 체인 구성
chain_with_summarized = RunnableWithMessageHistory(
    chain_legacy,
    get_summarized_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 첫 번째 질문
response = chain_with_summarized.invoke(
    {"input": "서울에서 가볼만한 곳을 추천해주세요."},
    config={"configurable": {"session_id": "tourist_summarized_1"}}
)

print("여행 가이드 답변:")
print(response.content)

현재 메시지 수: 2
여행 가이드 답변:
서울은 역사와 현대가 어우러진 도시로 다양한 볼거리와 즐길 거리가 많습니다. 아래 추천드리는 곳들을 참고하세요!

1. 경복궁
- 조선시대의 대표 궁궐로, 한국 전통 건축과 역사를 느낄 수 있습니다.
- 매일 오전 10시부터 오후 5시까지 개장하며, 수문장 교대식을 관람할 수 있습니다.

2. 북촌 한옥마을
- 전통 한옥이 잘 보존된 마을로, 고즈넉한 분위기와 함께 전통 문화 체험이 가능합니다.
- 산책하며 사진 찍기 좋은 곳입니다.

3. 명동
- 서울의 대표 쇼핑 거리로, 패션, 화장품, 길거리 음식 등 다양한 즐길 거리가 있습니다.
- 밤에는 화려한 네온사인과 거리 공연을 즐기실 수 있습니다.

4. 남산서울타워
- 서울의 랜드마크로, 케이블카를 타고 올라가면 도시 전경이 한눈에 들어옵니다.
- 야경이 특히 아름다우니 저녁 시간에 방문하는 것도 추천드립니다.

5. 인사동
- 전통과 현대가 어우러진 거리로, 전통 공예품, 갤러리, 찻집이 많습니다.
- 한국 전통 문화를 체험하기 좋은 곳입니다.

6. 동대문 디자인 플라자(DDP)
- 미래지향적 건축물로, 패션과 디자인 관련 전시, 시장, 이벤트가 열립니다.
- 야경이 멋지며, 주변 동대문 시장도 함께 둘러보세요.

7. 한강공원
- 강변을 따라 조성된 공원으로, 자전거 타기, 산책, 피크닉이 가능합니다.
- 여름철에는 한강 유람선도 즐기실 수 있습니다.

이 외에도 서울에는 다양한 박물관, 미술관, 카페, 맛집이 많으니 일정에 맞게 즐기시길 바랍니다. 궁금한 점이 있거나 더 구체적인 추천이 필요하시면 언제든 말씀해 주세요!


In [27]:
# 두 번째 질문
response = chain_with_summarized.invoke(
    {"input": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"},
    config={"configurable": {"session_id": "tourist_summarized_1"}}
)

print("여행 가이드 답변:")
print(response.content)

현재 메시지 수: 4
여행 가이드 답변:
가장 인기 있는 곳은 바로 **경복궁**입니다. 

경복궁은 한국의 대표적인 역사 유적지로, 조선시대의 왕궁이자 서울의 상징적인 명소입니다. 아름다운 전통 건축물과 함께 수문장 교대식, 궁궐 내부 투어 등 다양한 체험 프로그램이 있어 많은 관광객들이 방문합니다. 또한, 주변에는 북촌 한옥마을과 인사동도 가까워 함께 둘러보면 더욱 풍성한 여행이 될 것입니다.

이곳은 한국의 역사와 문화를 깊이 느낄 수 있는 장소로, 국내외 관광객 모두에게 사랑받는 명소입니다. 방문하시면 잊지 못할 추억을 만들 수 있을 거예요!


In [28]:
# 세 번째 질문 - 요약이 트리거됨 (6개 메시지 초과)
response = chain_with_summarized.invoke(
    {"input": "부산에서도 추천해주세요."},
    config={"configurable": {"session_id": "tourist_summarized_1"}}
)

print("여행 가이드 답변:")
print(response.content)

# 요약된 히스토리 확인
print("\n[요약된 대화 히스토리]")
history = get_summarized_history("tourist_summarized_1")
for i, msg in enumerate(history.messages):
    role = msg.__class__.__name__.replace("Message", "")
    content = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"{i+1}. [{role}]: {content}")

현재 메시지 수: 6
→ 대화가 요약되었습니다.
여행 가이드 답변:
물론입니다! 부산은 해변과 도시가 어우러진 매력적인 도시로, 다양한 관광 명소가 있습니다. 아래 추천드리는 곳들을 참고하세요.

1. 해운대 해수욕장
- 부산의 대표 해변으로, 맑은 바닷물과 넓은 백사장이 인상적입니다.
- 여름철에는 해수욕과 해변 스포츠를 즐기실 수 있으며, 주변에는 맛집과 카페도 많습니다.

2. 감천문화마을
- 알록달록한 집들과 예술 작품이 가득한 마을로, 사진 찍기 좋은 포토 스팟입니다.
- 지역 예술가들의 작품과 문화 체험도 가능합니다.

3. 부산 타워 (용두산공원)
- 부산 시내를 내려다볼 수 있는 전망대로, 야경이 특히 아름답습니다.
- 공원 내에는 다양한 조각상과 산책로도 있어 여유로운 시간을 보내기 좋아요.

4. 자갈치시장
- 부산의 대표 수산시장으로, 신선한 해산물과 다양한 먹거리를 즐기실 수 있습니다.
- 시장 구경과 함께 싱싱한 해산물 요리도 맛보세요.

5. 광안리 해수욕장
- 밤에는 광안대교의 화려한 조명이 펼쳐져 낭만적인 분위기를 자아냅니다.
- 해변 근처에는 카페와 레스토랑이 많아 여유로운 시간 보내기에 좋습니다.

6. 부산 영화의 전당 & BIFF 광장
- 부산 국제영화제(BIFF)의 중심지로, 영화와 문화 행사를 즐기실 수 있습니다.
- 현대적인 건축물과 다양한 문화 콘텐츠가 매력적입니다.

부산은 바다와 도시의 매력을 동시에 느낄 수 있는 곳이니, 일정에 맞게 방문하시면 멋진 추억을 만들 수 있을 거예요! 궁금한 점이나 더 구체적인 추천이 필요하시면 언제든 말씀해 주세요.

[요약된 대화 히스토리]
1. [AI]: 서울에서 가볼 만한 인기 명소로는 경복궁이 가장 추천됩니다. 경복궁은 조선시대 대표 궁궐로 전통 건축과 역사를 체험할 수 있으며, 수문장 교대식...
2. [Human]: 부산에서도 추천해주세요.
3. [AI]: 물론입니다! 부산은 해변과 도시가 어우러진 매력적인 도시로, 다양한 관광 명소가 있습니다. 아래 추천드리는 곳들을 참

---

## 3. `checkpointer` 파라미터를 활용한 메모리 관리 (권장 방식)

- LangChain 1.0에서는 `create_agent`가 에이전트 생성의 표준 방식입니다. `checkpointer` 파라미터를 통해 대화 이력을 자동으로 관리할 수 있습니다.

    - **간결한 코드**: StateGraph를 직접 구성할 필요 없이 몇 줄로 에이전트 생성
    - **자동 메모리 관리**: checkpointer가 thread_id별로 대화 내용을 자동 저장/복원
    
- **다양한 저장소 지원**: InMemorySaver, SqliteSaver, PostgresSaver 등

    | Checkpointer | 패키지 | 용도 |
    |-------------|--------|------|
    | `InMemorySaver` | `langgraph` (내장) | 개발/테스트용, 휘발성 |
    | `SqliteSaver` | `langgraph-checkpoint-sqlite` | 로컬 영구 저장 |
    | `PostgresSaver` | `langgraph-checkpoint-postgres` | 프로덕션 환경 |

### 3.1 InMemorySaver (메모리 기반)

* `InMemorySaver`는 가장 간단한 형태의 checkpointer로, 메모리에 상태를 저장합니다.
* 개발 및 테스트 환경에서 빠르게 프로토타이핑할 때 유용합니다.
* 프로세스가 종료되면 저장된 데이터가 사라지므로 프로덕션 환경에는 적합하지 않습니다.

* **create_agent vs StateGraph 비교:**

    ```python
    # StateGraph 방식 (약 20줄)
    workflow = StateGraph(MessagesState)
    workflow.add_node("call_model", call_model)
    workflow.add_edge(START, "call_model")
    workflow.add_edge("call_model", END)
    app = workflow.compile(checkpointer=checkpointer)

    # create_agent 방식 (약 5줄)
    agent = create_agent(
        model="gpt-4.1-nano",
        system_prompt="당신은 여행 가이드입니다.",
        checkpointer=InMemorySaver()
    )
    ```

In [29]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# create_agent를 사용하여 간단하게 대화형 에이전트 생성
agent = create_agent(
    model="gpt-4.1-nano",  # 사용할 모델
    tools=[],  # 도구 없이 순수 대화형 에이전트
    system_prompt="당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요.",
    checkpointer=InMemorySaver(),  # 메모리 기반 checkpointer
)

print("create_agent 기반 챗봇이 준비되었습니다.")

create_agent 기반 챗봇이 준비되었습니다.


In [35]:
# thread_id를 사용하여 대화 세션 구분
config = {"configurable": {"thread_id": "tourist_1"}, "callbacks":[langfuse_handler]}

# 첫 번째 질문
response = agent.invoke(
    {"messages": [{"role": "user", "content": "서울에서 가볼만한 곳을 추천해주세요."}]},
    config=config
)

print("여행 가이드 답변:")
print(response["messages"][-1].content)

여행 가이드 답변:
서울에서 방문하기 좋은 명소를 여러 곳 추천드릴게요! 다양한 매력을 느끼실 수 있는 곳들입니다.

1. 경복궁과 광화문광장  
한국의 전통과 역사를 느낄 수 있는 대표 궁궐입니다. 수문장 교대식과 궁궐 내부 구경도 놓치지 마세요.

2. 북촌 한옥마을  
전통 한옥이 잘 보존된 곳으로, 조용한 골목길 산책과 사진 촬영에 좋아요. 한국의 옛 정취를 만끽할 수 있습니다.

3. N서울타워 (남산타워)  
서울의 시내 전경을 한눈에 내려다볼 수 있는 전망대. 케이블카를 타고 오르는 재미도 쏠쏠합니다.

4. 명동과 쇼핑거리  
다양한 패션과 먹거리, 화려한 야경을 즐기실 수 있는 쇼핑 명소입니다. 거리 음식을 맛보는 것도 추천드려요.

5. 홍대와 이태원  
젊음과 트렌드의 거리로, 독특한 카페, 공연, 다문화 음식점이 가득합니다. 문화와 예술을 즐기기에 완벽한 곳이죠.

6. 서울숲과 한강공원  
도심 속 자연을 만끽할 수 있는 공간으로, 산책이나 자전거 타기, 피크닉을 즐기실 수 있어요.

7. 동대문과 DDP (동대문 디자인 플라자)  
패션과 디자인의 중심지로, 야간 조명이 아름답고 야시장도 열립니다.

이 외에도 다양한 매력이 가득한 서울! 어떤 테마나 관심사가 있으시면 더욱 맞춤 추천도 도와드릴 수 있어요. 즐거운 여행 되시길 바랍니다!


In [31]:
# 대화 히스토리 확인 (현재 상태의 모든 메시지)
print("현재 대화 히스토리:")
for msg in response["messages"]:
    msg.pretty_print()

현재 대화 히스토리:
================================ Human Message =================================

서울에서 가볼만한 곳을 추천해주세요.
================================== Ai Message ==================================

서울에서 방문하기 좋은 곳은 정말 다양합니다! 아래 추천 드리는 곳들을 참고하시면 즐거운 여행이 되실 거예요.

1. 경복궁과 광화문광장  
조선시대의 대표 궁궐인 경복궁은 한국 전통 건축의 아름다움을 느낄 수 있는 곳입니다. 근처 광화문광장에서 사진도 찍고, 수문장 교대식도 감상하세요.

2. 북촌 한옥마을  
전통 한옥이 잘 보존된 마을로, 고즈넉한 분위기 속에서 한국의 옛 정취를 느낄 수 있습니다. 골목길 산책하며 사진도 찍기 좋아요.

3. N서울타워 (남산타워)  
서울 시내 전경을 한눈에 내려다볼 수 있는 전망대입니다. 케이블카를 타고 올라가거나 남산 공원 산책도 함께 즐기실 수 있어요.

4. 명동과 쇼핑거리  
국제적 브랜드부터 한국 로컬 브랜드까지 다양한 쇼핑과 맛집이 모여있는 곳입니다. 거리의 활기와 길거리 음식도 체험해보세요.

5. 홍대와 이태원  
젊음의 거리인 홍대에는 독특한 카페와 예술적 공간, 라이브 공연장이 많고, 이태원은 다문화적인 분위기와 다양한 세계 요리를 맛볼 수 있는 곳입니다.

6. 서울숲과 한강공원  
도심 속에서 자연을 만끽할 수 있는 곳입니다. 산책, 자전거 타기, 피크닉 등을 즐기실 수 있어요.

7. 동대문 and Dongdaemun Design Plaza (DDP)  
패션과 디자인의 중심지로, 야간 조명이 아름다운 DDP와 야시장도 유명합니다.

가이드드 투어나 각 관광지에 대한 자세한 정보와 추천 일정도 도와드릴 수 있으니 필요하시면 언제든 말씀하세요!


In [32]:
# 같은 thread_id로 후속 질문 - 이전 대화를 기억함
response = agent.invoke(
    {"messages": [{"role": "user", "content": "이전에 추천한 장소 중에서 가장 인기 있는 곳은 어디인가요?"}]},
    config=config
)

print("여행 가이드 답변:")
print(response["messages"][-1].content)

여행 가이드 답변:
가장 인기 있는 장소는 역시 **경복궁과 N서울타워**입니다.  

- **경복궁**은 한국 전통 건축과 역사를 만날 수 있는 대표적인 명소로, 국내외 관광객 모두에게 매우 인기가 높습니다. 특히, 수문장 교대식과 궁궐 내부 투어는 꼭 경험해보시기를 추천드려요.  

- **N서울타워**는 서울의 전경을 한눈에 내려다볼 수 있는 곳으로, 특히 야경이 아름다워 많은 여행객들이 방문합니다. 케이블카를 타고 올라가는 것도 즐거운 경험입니다.  

이 두 곳은 서울의 상징적이고 가장 많은 사람들이 찾는 인기 명소입니다. 방문 계획 세우실 때 참고하시면 좋을 것 같아요!


In [33]:
# 대화 히스토리 확인
print(f"총 메시지 수: {len(response['messages'])}")
print("\n대화 히스토리:")
for msg in response["messages"]:
    msg.pretty_print()

총 메시지 수: 4

대화 히스토리:
================================ Human Message =================================

서울에서 가볼만한 곳을 추천해주세요.
================================== Ai Message ==================================

서울에서 방문하기 좋은 곳은 정말 다양합니다! 아래 추천 드리는 곳들을 참고하시면 즐거운 여행이 되실 거예요.

1. 경복궁과 광화문광장  
조선시대의 대표 궁궐인 경복궁은 한국 전통 건축의 아름다움을 느낄 수 있는 곳입니다. 근처 광화문광장에서 사진도 찍고, 수문장 교대식도 감상하세요.

2. 북촌 한옥마을  
전통 한옥이 잘 보존된 마을로, 고즈넉한 분위기 속에서 한국의 옛 정취를 느낄 수 있습니다. 골목길 산책하며 사진도 찍기 좋아요.

3. N서울타워 (남산타워)  
서울 시내 전경을 한눈에 내려다볼 수 있는 전망대입니다. 케이블카를 타고 올라가거나 남산 공원 산책도 함께 즐기실 수 있어요.

4. 명동과 쇼핑거리  
국제적 브랜드부터 한국 로컬 브랜드까지 다양한 쇼핑과 맛집이 모여있는 곳입니다. 거리의 활기와 길거리 음식도 체험해보세요.

5. 홍대와 이태원  
젊음의 거리인 홍대에는 독특한 카페와 예술적 공간, 라이브 공연장이 많고, 이태원은 다문화적인 분위기와 다양한 세계 요리를 맛볼 수 있는 곳입니다.

6. 서울숲과 한강공원  
도심 속에서 자연을 만끽할 수 있는 곳입니다. 산책, 자전거 타기, 피크닉 등을 즐기실 수 있어요.

7. 동대문 and Dongdaemun Design Plaza (DDP)  
패션과 디자인의 중심지로, 야간 조명이 아름다운 DDP와 야시장도 유명합니다.

가이드드 투어나 각 관광지에 대한 자세한 정보와 추천 일정도 도와드릴 수 있으니 필요하시면 언제든 말씀하세요!
================================ Human Message ====

In [34]:
# 다른 thread_id로 새로운 대화 시작 - 이전 대화와 독립적
config_new = {"configurable": {"thread_id": "tourist_2"}}

response_new = agent.invoke(
    {"messages": [{"role": "user", "content": "부산에서 맛집을 추천해주세요."}]},
    config=config_new
)

print("[새로운 대화 세션]")
print("여행 가이드 답변:")
print(response_new["messages"][-1].content)

[새로운 대화 세션]
여행 가이드 답변:
부산은 해산물과 맛집이 가득한 도시로 유명합니다. 여행객들에게 인기 있는 맛집들을 소개해드릴게요.

1. 광안리 해변 근처 - **금수복국**  
신선한 복어요리와 해산물 요리를 맛볼 수 있는 곳으로, 부산 대표 맛집입니다. 복국외에도 복지리와 복어요리 세트가 인기입니다.

2. 자갈치 시장 - **자갈치시장 내 다양한 해산물 맛집**  
신선한 해산물을 바로 구매하거나, 시장 내 다양한 식당에서 회, 해물탕, 구이 등을 즐기실 수 있습니다. 부산의 활기찬 시장 분위기를 경험하세요.

3. 해운대구 - **밀면골목**  
부산 특유의 냉면인 밀면을 맛볼 수 있는 곳들이 모여있습니다. 더운 여름철에 시원한 밀면 한 그릇 추천드려요.

4. 남포동 - **범일국밥집**  
부산 돼지국밥과 다양한 한식을 맛볼 수 있습니다. 지역 주민들도 즐겨 찾는 곳입니다.

5. 기장군 - **대변항의 회센터들**  
싱싱한 회와 해산물 구이를 즐기기에 좋은 곳입니다. 바닷가 풍경과 함께 신선한 해산물을 맛보세요.

추가로, 부산은 감천문화마을이나 태종대, 부산타워 등 관광명소와 함께 다양한 맛집을 즐기실 수 있으니, 방문 계획에 참고하세요. 즐거운 여행 되시길 바랍니다!


### 3.2 SqliteSaver (영구 저장소)

* `SqliteSaver`는 SQLite 데이터베이스를 사용하여 대화 상태를 영구적으로 저장합니다.
* 프로세스가 재시작되어도 이전 대화를 복원할 수 있습니다.
* 로컬 개발 환경이나 단일 서버 환경에서 사용하기 적합합니다.

* **설치:**
    ```bash
    uv add langgraph-checkpoint-sqlite
    ```

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. 'with' 문을 사용하여 체크포인터를 생성합니다.
with SqliteSaver.from_conn_string("chat_history.db") as sqlite_checkpointer:
    
    # 2. 'with' 블록 안에서 에이전트를 생성하고 사용해야 합니다.
    agent_sqlite = create_agent(
        model="gpt-4.1-nano",
        tools=[],
        system_prompt="당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요.",
        checkpointer=sqlite_checkpointer, # 이제 올바른 객체가 전달됩니다.
    )

    print("SQLite 기반 챗봇이 준비되었습니다.")
    
    # 3. 에이전트 실행 예시
    # config = {"configurable": {"thread_id": "user_123"}}
    # response = agent_sqlite.invoke({"messages": [("user", "서울의 명소는?")]}, config)
    # print(response)

# 'with' 블록을 벗어나면 DB 연결이 자동으로 안전하게 닫힙니다.

In [ ]:
# SQLite 기반 대화 시작
with SqliteSaver.from_conn_string("chat_history.db") as sqlite_checkpointer:

    agent_sqlite = create_agent(
        model="gpt-4.1-nano",
        tools=[],
        system_prompt="당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요.",
        checkpointer=sqlite_checkpointer, # 이제 올바른 객체가 전달됩니다.
    )

    config_sqlite = {"configurable": {"thread_id": "sqlite_tourist_1"}}

    response = agent_sqlite.invoke(
        {"messages": [{"role": "user", "content": "제주도에서 꼭 가봐야 할 곳을 추천해주세요."}]},
        config=config_sqlite
    )

    print("여행 가이드 답변:")
    print(response["messages"][-1].content)

In [ ]:
# 후속 질문 - SQLite에서 이전 대화를 불러옴
with SqliteSaver.from_conn_string("chat_history.db") as sqlite_checkpointer:

    agent_sqlite = create_agent(
        model="gpt-4.1-nano",
        tools=[],
        system_prompt="당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요.",
        checkpointer=sqlite_checkpointer, # 이제 올바른 객체가 전달됩니다.
    )

    config_sqlite = {"configurable": {"thread_id": "sqlite_tourist_1"}}

    response = agent_sqlite.invoke(
        {"messages": [{"role": "user", "content": "그 중에서 아이와 함께 가기 좋은 곳은 어디인가요?"}]},
        config=config_sqlite
    )

    print("여행 가이드 답변:")
    print(response["messages"][-1].content)

In [ ]:
# 이전 대화 내역 조회
with SqliteSaver.from_conn_string("chat_history.db") as sqlite_checkpointer:
    
    # 1. 에이전트 생성
    agent_sqlite = create_agent(
        model="gpt-4.1-nano",
        tools=[],
        system_prompt="당신은 여행 가이드입니다. 관광객에게 유용한 정보를 제공하세요.",
        checkpointer=sqlite_checkpointer, # 이제 올바른 객체가 전달됩니다.
    )

    # 2. 불러오고 싶은 특정 thread_id 설정
    # 이 ID가 같으면 DB에서 이전 상태(State)를 자동으로 찾아옵니다.
    config = {"configurable": {"thread_id": "sqlite_tourist_1"}}

    # 3. [선택 사항] 이전 대화 내용이 잘 들어있는지 확인하기
    current_state = agent_sqlite.get_state(config)
    if current_state.values:
        print("--- 이전 대화 기록 발견 ---")
        for message in current_state.values.get("messages", []):
            role = "Human" if message.type == "human" else "AI"
            print(f"[{role}]: {message.content[:50]}...")
    else:
        print("--- 새로운 대화 세션입니다 ---")

---
# **[실습]**

- 메시지 트리밍과 대화 요약 저장을 결합하여 메시지를 관리하는 기능을 구현합니다. 

- **힌트**
    1. `TrimmedInMemoryHistory`의 `add_messages` 메서드를 참고하세요
    2. 트리밍 전에 제거될 메시지들을 먼저 식별하세요
    3. 제거될 메시지들에 대해서만 요약을 생성하세요
    4. 요약 메시지는 `summarized_messages` 리스트에 추가하세요

- **구현 순서:**
    1. 새 메시지 추가
    2. 현재 메시지 수가 `max_tokens` 초과 확인
    3. 초과 시, 제거될 메시지 식별
    4. 제거될 메시지 요약 생성
    5. 요약을 `summarized_messages`에 추가
    6. 현재 메시지는 트리밍 적용

In [ ]:
class TrimmedAndSummarizedHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage]
    max_tokens: int  # 트리밍 기준
    summary_threshold: int  # 요약 기준
    llm: ChatOpenAI
    summarized_messages: List[BaseMessage]  # 요약된 메시지 저장용

2. 메시지 처리 로직
- 새 메시지 추가시 max_tokens 체크
- 트리밍 발생하면 제거될 메시지 식별
- 제거 예정 메시지들은 요약하여 summarized_messages에 저장
- 현재 메시지는 트리밍된 상태로 유지

3. 요약 프로세스
- 트리밍으로 제거될 메시지들만 선별
- 선별된 메시지들에 대해 summary_chain 실행
- 요약본을 시스템 메시지로 변환하여 저장

In [ ]:
# 여기에 코드를 작성하세요.